# Atributos das imagens com CLIP (Kaggle, GPU T4)

Roda o mesmo `atributos_clip.py` do repositorio em todas as pecas da FashionVerse.
Antes: suba a pasta `preprocess` como dataset com o nome `recomendador-codigo`, ligue GPU T4 e internet.
Ao final, baixe `/kaggle/working/atributos_clip.csv`.

In [ ]:
!pip install -q open_clip_torch

In [ ]:
import csv, glob, os, subprocess, sys
from pathlib import Path

# pasta do dataset com preprocessar.py, aplicacao.py, referencia.py e atributos_clip.py
CODIGO = Path(glob.glob("/kaggle/input/**/atributos_clip.py", recursive=True)[0]).parent
sys.path.insert(0, str(CODIGO))
from imagens import URL
from preprocessar import carregar_pecas
print(CODIGO)

In [ ]:
# o zip tem 3,6 GB. /kaggle/working guarda so 20 GB, entao tudo fica em /tmp
!curl -s -L -o /tmp/fv.zip "{URL}"
!unzip -q -o /tmp/fv.zip -d /tmp/fv && rm /tmp/fv.zip
!du -sh /tmp/fv

In [ ]:
# o json pode vir no zip ou junto com o codigo
achados = glob.glob("/tmp/fv/**/Combined_Metadata.json", recursive=True) + glob.glob(f"{CODIGO}/**/Combined_Metadata.json", recursive=True)
pecas = carregar_pecas(achados[0])

# mesma ordem do preprocessar.py, entao o id e a posicao na lista
raiz = Path(glob.glob("/tmp/fv/**/FashionVerse-MME", recursive=True)[0])
porid = Path("/tmp/porid")
porid.mkdir(exist_ok=True)
faltam = 0
with open("/kaggle/working/pecas_categorias.csv", "w", newline="") as arq:
    esc = csv.writer(arq, lineterminator="\n")
    esc.writerow(["id", "categoria"])
    for i, p in enumerate(pecas):
        esc.writerow([i, p["categoria"]])
        origem = raiz / p["imagem"]
        if not origem.exists():
            faltam += 1
            continue
        destino = porid / f"{i}.jpg"
        if not destino.exists():
            destino.symlink_to(origem)
print(len(pecas), "pecas,", faltam, "sem imagem")

In [ ]:
# se o pecas.csv do repositorio subiu junto, confere se os ids batem
antigo = glob.glob(f"{CODIGO}/**/pecas.csv", recursive=True)
if antigo:
    with open(antigo[0]) as a, open("/kaggle/working/pecas_categorias.csv") as b:
        assert [l.split(",")[:2] for l in a] == [l.strip().split(",")[:2] for l in b], "ids diferentes"
    print("ids iguais aos do pecas.csv")

In [ ]:
!python {CODIGO}/atributos_clip.py --imagens /tmp/porid --pecas /kaggle/working/pecas_categorias.csv --saida /kaggle/working/atributos_clip.csv --lote 256 --leitores 4